# Vanilla autoencoder

Dense reconstruction baseline for Q1 and Figures 1 and 4. The only training signal is pixel MSE, so the decoder learns the conditional average and the results tend to look soft.

Run this on a Colab **GPU** runtime. The project folder must sit at `MyDrive/dl-sesi07`. Change `DRIVE_ROOT` in the setup cell if your folder name differs. Run the cells from top to bottom. Results are written to `artifacts/` on Drive.


In [ ]:
import os
import sys
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/dl-sesi07"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path(DRIVE_ROOT)
else:
    ROOT = Path.cwd().resolve()
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if "google.colab" in sys.modules:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])


## Setup

The network is `784 → 256 → 64 → 256 → 784` with ReLU and a sigmoid output. Pixels stay in `[0, 1]`.


In [ ]:
%matplotlib inline
import time

import torch
import torch.nn.functional as F
from torch.optim import Adam
from tqdm.auto import tqdm

from src.config import (
    ALPHAS,
    CLASS_NAMES,
    LATENT_DIM,
    NEAR_ZERO,
    SEED,
    SPARSITY_LAMBDA,
    get_device,
    set_seed,
)
from src.data import class_pair, fixed_test_images, get_splits, make_loader
from src.io import load_checkpoint, load_json, save_json, save_shared, save_torch
from src.metrics import activation_summary, measure_latency_ms, reconstruction_scores, stability_from_curve
from src.models import VanillaAE
from src.plots import show_grid

set_seed(SEED)
DEVICE = get_device()
EPOCHS = 30
NAME = "vanilla_ae"
print("device", DEVICE)

train_set, _, test_set = get_splits()
test_loader = make_loader(test_set, shuffle=False)
test_images, test_labels = fixed_test_images()
save_shared("test8.pt", {"images": test_images, "labels": test_labels})
print("fixed test labels", [CLASS_NAMES[i] for i in test_labels])

def latent_strip(decode, z_a, z_b):
    frames = []
    for alpha in ALPHAS:
        frames.append(decode((1 - alpha) * z_a + alpha * z_b).cpu())
    return torch.cat(frames)


## Loss

Mean pixel MSE between the sigmoid output and the image. There is no penalty on the 64-d code.


In [ ]:
model = VanillaAE().to(DEVICE)
optimizer = Adam(model.parameters(), lr=1e-3)
history = {"recon_loss": [], "epoch_seconds": []}

for epoch in range(EPOCHS):
    set_seed(SEED + epoch)
    loader = make_loader(train_set, shuffle=True, seed=SEED + epoch)
    model.train()
    total = 0.0
    seen = 0
    pass
    start = time.perf_counter()
    for images, _ in tqdm(loader, leave=False):
        images = images.to(DEVICE)
        restored, latent = model(images)
        mse = F.mse_loss(restored, images)
        loss = mse
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        pixel_sum = F.mse_loss(restored, images, reduction="sum").item()
        total += pixel_sum
        seen += images.numel()
        pass
    elapsed = time.perf_counter() - start
    history["recon_loss"].append(total / seen)
    history["epoch_seconds"].append(elapsed)
    print(f"epoch {epoch + 1:02d}  mse {total / seen:.4f}  {elapsed:.1f}s")

save_json(NAME, "history.json", history)
save_torch(NAME, "checkpoint.pt", {"model": model.state_dict(), "epoch": EPOCHS - 1})


## Test scores and the saved strips

Test MSE, PSNR, and SSIM use the full test split. The eight reconstructions and the class-0 to class-9 interpolation are what notebook 08 draws. Latency is one full forward of 64 images.


In [ ]:
checkpoint = load_checkpoint(NAME, map_location=DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()
history = load_json(NAME, "history.json")

def reconstruct(images):
    restored, _ = model(images)
    return restored

scores = reconstruction_scores(reconstruct, test_loader, DEVICE)
latents = []
with torch.no_grad():
    for images, _ in test_loader:
        _, latent = model(images.to(DEVICE))
        latents.append(latent.cpu())
summary = activation_summary(torch.cat(latents))

batch = torch.rand(64, 1, 28, 28, device=DEVICE)
latency = measure_latency_ms(lambda: model(batch), DEVICE)
preview, _ = model(test_images.to(DEVICE))
preview = preview.cpu()

image_a, image_b = class_pair()
with torch.no_grad():
    z_a = model.encode(image_a.to(DEVICE))
    z_b = model.encode(image_b.to(DEVICE))
    strip = latent_strip(model.decode, z_a, z_b)

save_torch(NAME, "recon.pt", preview)
save_torch(NAME, "interp.pt", strip)
save_json(
    NAME,
    "metrics.json",
    {
        "latent_dim": LATENT_DIM,
        "sec_per_epoch": sum(history["epoch_seconds"]) / len(history["epoch_seconds"]),
        "latency_ms": latency,
        "test_mse": scores["test_mse"],
        "test_psnr": scores["test_psnr"],
        "test_ssim": scores["test_ssim"],
        "stability": stability_from_curve(history["recon_loss"]),
        **summary,
    },
)
show_grid(torch.cat([test_images, preview]), nrow=8, title="Originals, then reconstructions")
show_grid(strip, nrow=6, title="Latent interpolation, alpha 0 to 1")
print({key: round(value, 4) if isinstance(value, float) else value for key, value in load_json(NAME, "metrics.json").items()})
